In [1]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Long-Horizon Agent Memory Demo

<table align="left">

  <td>
    <a href="https://colab.research.google.com/github/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/memory_service_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/colab-logo.png" alt="Colab logo"> Run in Colab
    </a>
  </td>
  <td>
    <a href="https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/memory_service_demo.ipynb">
      <img src="https://raw.githubusercontent.com/googleapis/python-bigquery-dataframes/refs/heads/main/third_party/logo/github-logo.png" width="32" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/main/examples/memory_service_demo.ipynb">
      <img src="https://www.gstatic.com/images/branding/product/1x/google_cloud_48dp.png" alt="Vertex AI logo" width="32">
      Open in Vertex AI Workbench
    </a>
  </td>
  <td>
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/haiyuan-eng-google/BigQuery-Agent-Analytics-SDK/blob/main/examples/memory_service_demo.ipynb">
      <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTW1gvOovVlbZAIZylUtf5Iu8-693qS1w5NJw&s" alt="BQ logo" width="35">
      Open in BQ Studio
    </a>
  </td>
</table>

Give agents memory across sessions using historical BigQuery trace data. This notebook demonstrates cross-session context retrieval, semantic memory search, user profile building, and context management.

## Install Dependencies

In [2]:
!pip install -q bigquery-agent-analytics google-cloud-bigquery nest-asyncio

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


ERROR: Could not find a version that satisfies the requirement bigquery-agent-analytics (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /usr/local/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip
ERROR: No matching distribution found for bigquery-agent-analytics


## Authenticate & Configure

In [3]:
import os

# Colab authentication
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Colab authentication successful.")
except ImportError:
    print("Not running in Colab — using default credentials.")

# ---------- Configuration ----------
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "test-project-0728-467323")
DATASET_ID = os.environ.get("BQ_DATASET", "agent_analytics")
TABLE_ID = os.environ.get("BQ_TABLE", "agent_events")
LOCATION = "US"

# Enable async in Jupyter
import nest_asyncio
nest_asyncio.apply()

print(f"Project  : {PROJECT_ID}")
print(f"Dataset  : {DATASET_ID}")
print(f"Table    : {TABLE_ID}")

Not running in Colab — using default credentials.
Project  : test-project-0728-467323
Dataset  : agent_analytics
Table    : agent_events


## 1. Cross-Session Context Retrieval

`BigQueryMemoryService` retrieves relevant past interactions for a user, providing continuity across agent sessions. This is the foundation for agents that remember past conversations.

In [4]:
import asyncio
from bigquery_agent_analytics import BigQueryMemoryService

memory = BigQueryMemoryService(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
)

episodes = []  # initialise so later cells never hit NameError

print("[Cross-Session Context Retrieval]")
try:
    # Retrieve past episodes for a user
    episodes = asyncio.get_event_loop().run_until_complete(
        memory.get_session_context(
            user_id="demo_user",
            current_session_id="current-session",
            lookback_sessions=5,
        )
    )

    if episodes:
        print(f"Retrieved {len(episodes)} past episodes:\n")
        for i, ep in enumerate(episodes, 1):
            print(f"  Episode {i}:")
            print(f"    Timestamp    : {ep.timestamp}")
            print(f"    User message : {ep.user_message[:100]}...")
            print(f"    Agent response: {ep.agent_response[:100]}...")
            print(f"    Tool calls   : {ep.tool_calls}")
            print()
    else:
        print("  No past episodes found for this user.")
        print("  (Run the e2e_notebook_demo.ipynb first to generate trace data)")
except Exception as exc:
    print(f"  Context retrieval failed: {exc}")

[Cross-Session Context Retrieval]


Retrieved 5 past episodes:

  Episode 1:
    Timestamp    : 2026-04-03 07:44:56.141231+00:00
    User message : Now find hotels in Paris checking in 2025-04-20 and checking out 2025-04-25....
    Agent response: text: 'I found several hotel options in Paris for your stay from April 20 to April 25, 2025:

*   **...
    Tool calls   : []

  Episode 2:
    Timestamp    : 2026-04-03 07:44:09.552945+00:00
    User message : I want to plan a 5-day vacation to Tokyo from 2025-05-01 to 2025-05-06. Search flights from Los Ange...
    Agent response: text: 'OK! Here is a plan for your 5-day vacation to Tokyo:

### ✈️ Flights
The best flight option f...
    Tool calls   : []

  Episode 3:
    Timestamp    : 2026-04-03 07:43:59.242808+00:00
    User message : Plan a weekend trip from San Francisco to New York departing 2025-04-12 and returning 2025-04-14. Se...
    Agent response: text: 'Here is a proposed plan for your weekend trip from San Francisco to New York from **April 12 ...
    Tool calls

## 2. Semantic Memory Search

Search past episodes by semantic similarity — find relevant context even when exact keywords don't match.

In [5]:
print("[Semantic Memory Search]")
try:
    results = asyncio.get_event_loop().run_until_complete(
        memory.search_memory(
            app_name="e2e_notebook_demo",
            user_id="demo_user",
            query="How do I plan a trip to Tokyo?",
        )
    )

    if hasattr(results, 'memories') and results.memories:
        print(f"Found {len(results.memories)} relevant memories:\n")
        for entry in results.memories:
            print(f"  Key  : {entry.key}")
            print(f"  Value: {entry.value[:200]}...")
            print()
    else:
        print("  No matching memories found.")
        print("  (Semantic search requires embeddings — run trace generation first)")
except Exception as exc:
    print(f"  Memory search failed: {exc}")

[Semantic Memory Search]


Found 6 relevant memories:

  Memory search failed: 'MemoryEntry' object has no attribute 'key'


## 3. User Profile Building

`UserProfileBuilder` aggregates a user's interaction history into a structured profile — topics of interest, communication style, common requests, and preferred tools.

In [6]:
from bigquery_agent_analytics import UserProfileBuilder

builder = UserProfileBuilder(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    table_id=TABLE_ID,
)

print("[User Profile Building]")
try:
    profile = asyncio.get_event_loop().run_until_complete(
        builder.build_profile(user_id="demo_user")
    )

    print(f"Profile for user: demo_user\n")
    if hasattr(profile, 'topics_of_interest'):
        print(f"  Topics of interest    : {profile.topics_of_interest}")
    if hasattr(profile, 'communication_style'):
        print(f"  Communication style   : {profile.communication_style}")
    if hasattr(profile, 'common_requests'):
        print(f"  Common requests       : {profile.common_requests}")
    if hasattr(profile, 'preferred_tools'):
        print(f"  Preferred tools       : {profile.preferred_tools}")
    if hasattr(profile, 'session_count'):
        print(f"  Total sessions        : {profile.session_count}")
except Exception as exc:
    print(f"  Profile building failed: {exc}")

[User Profile Building]


Failed to extract patterns with LLM: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.


Profile for user: demo_user

  Topics of interest    : []
  Communication style   : None
  Common requests       : []
  Preferred tools       : ['search_flights', 'calculate_trip_budget', 'search_hotels', 'get_weather_forecast']
  Total sessions        : 6


## 4. Context Management

`ContextManager` prevents cognitive overload by selecting only the most relevant memories within a token budget. It balances relevance and recency.

In [7]:
from bigquery_agent_analytics import ContextManager

ctx_mgr = ContextManager(
    max_context_tokens=32000,
    relevance_weight=0.7,
    recency_weight=0.3,
)

print("[Context Management]")
if episodes:
    # Select the most relevant memories within token budget
    relevant = ctx_mgr.select_relevant_context(
        current_task="How do I change my subscription plan?",
        available_memories=episodes,
        current_context_tokens=5000,
    )
    print(f"Selected {len(relevant)} relevant episodes "
          f"from {len(episodes)} available.\n")
    for ep in relevant:
        print(f"  [{ep.timestamp}] {ep.user_message[:80]}...")

    # Summarize older context to save tokens
    try:
        summary, recent = asyncio.get_event_loop().run_until_complete(
            ctx_mgr.summarize_old_context(
                memories=episodes,
                preserve_recent=3,
            )
        )
        print(f"\n  Context summary: {summary[:200]}...")
        print(f"  Preserved {len(recent)} recent episodes.")
    except Exception as exc:
        print(f"  Summarization failed: {exc}")
else:
    print("  No episodes available — run section 1 first.")
    print("  (ContextManager works with episodes from get_session_context)")

Failed to summarize context: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.


[Context Management]
Selected 5 relevant episodes from 5 available.

  [2026-04-03 07:44:09.552945+00:00] I want to plan a 5-day vacation to Tokyo from 2025-05-01 to 2025-05-06. Search f...
  [2026-04-03 07:37:13.697361+00:00] I want to plan a 5-day vacation to Tokyo from 2025-05-01 to 2025-05-06. Search f...
  [2026-04-03 07:44:56.141231+00:00] Now find hotels in Paris checking in 2025-04-20 and checking out 2025-04-25....
  [2026-04-03 07:43:59.242808+00:00] Plan a weekend trip from San Francisco to New York departing 2025-04-12 and retu...
  [2026-04-03 07:37:40.342588+00:00] Now find hotels in Paris checking in 2025-04-20 and checking out 2025-04-25....
  Summarization failed: 'NoneType' object is not subscriptable


---

## Summary

| Feature | Class | Description |
|---|---|---|
| Cross-Session Context | `BigQueryMemoryService` | Retrieve past user interactions across sessions |
| Semantic Search | `BigQueryMemoryService` | Find relevant memories by semantic similarity |
| User Profiles | `UserProfileBuilder` | Aggregate user history into structured profiles |
| Context Management | `ContextManager` | Select relevant memories within token budgets |

### Key Takeaway
Long-horizon memory transforms stateless agents into contextual assistants that remember user preferences, past interactions, and common patterns — all backed by BigQuery's scalable storage and AI.EMBED for semantic retrieval.

In [8]:
print("\nNotebook complete!")


Notebook complete!
